# Two-stage probabilistic forecasting

This notebook covers both supported scenarios: **Negative Binomial** for non-negative integer counts and **Gamma** for strictly positive continuous values. Each example shows the PPF, CDF, the applicable probability function, and Newsvendor optimization.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from mlforecast import MLForecast
from sklearn.linear_model import LinearRegression
import sys
import os
sys.path.append(os.path.abspath("../.."))
from tinyshift.modelling import (
    FirstStageForecasterEvaluator,
    GammaFamily,
    NewsvendorOptimizer,
    TwoStageForecasterEvaluator,
    TwoStageForecasterWrapper,
)

pd.set_option("display.max_columns", 20)

## 1. Discrete scenario: Negative Binomial

The default family models count data and supports `cdf`, `ppf`, and `pmf`. Its quantiles and optimized quantities are integers.

In [ ]:
def make_count_data(n_periods=180, seed=42):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2024-01-01", periods=n_periods, freq="D")
    frames = []
    for offset, unique_id in enumerate(["store_A", "store_B"]):
        mean = 2.5 + offset + np.linspace(0, 1, n_periods) + 1.2 * (dates.dayofweek >= 5)
        size = 4.0
        y = rng.negative_binomial(size, size / (size + mean))
        frames.append(pd.DataFrame({"unique_id": unique_id, "ds": dates, "y": y}))
    return pd.concat(frames, ignore_index=True)

count_data = make_count_data()
count_data.head()

In [ ]:
count_fcst = MLForecast(
    models=[LinearRegression()], freq="D", lags=[1, 7, 14], date_features=["dayofweek"]
)
count_model = TwoStageForecasterWrapper(count_fcst).fit(
    count_data, h=7, n_windows=4
)

### Discrete PPF, CDF, and PMF

`predict_distribution` returns the forecast frame and its row-aligned distribution. Evaluate `ppf` for quantiles, `cdf` for cumulative probabilities, and `pmf` for exact probabilities. A scalar is evaluated for every forecast row, while a grid returns one column per grid value.

In [ ]:
count_frame, count_distribution = count_model.predict_distribution(h=7)
count_ppf = count_distribution.ppf([0.50, 0.90, 0.95])

count_frame.assign(
    probability_y_le_5=count_distribution.cdf(5),
    median=count_ppf[:, 0],
    p90=count_ppf[:, 1],
    p95=count_ppf[:, 2],
).head()

In [ ]:
# Exact probabilities P(Y=k), plus the probability above max_k.
max_k = 8
count_units = np.arange(max_k + 1)
count_pmf = count_distribution.pmf(count_units)
count_probabilities = count_frame.copy()
for index, unit in enumerate(count_units):
    count_probabilities[f"P(Y={unit})"] = count_pmf[:, index]
count_probabilities[f"P(Y>{max_k})"] = 1.0 - count_pmf.sum(axis=1)
count_probabilities.head()

### Optimize discrete inventory

`NewsvendorOptimizer.optimize` computes the Newsvendor critical ratio `underage_cost / (underage_cost + overage_cost)` and evaluates the distribution PPF at that probability. Costs can be scalars, `cost_df` columns, or dictionaries.

In [ ]:
discrete_plan = NewsvendorOptimizer.optimize(
    count_frame,
    count_distribution,
    underage_cost=10.0,
    overage_cost=2.0,
)
discrete_plan.head()

## 2. Continuous scenario: Gamma

Pass `GammaFamily()` for continuous, strictly positive targets. Gamma supports `cdf` and `ppf`; 

In [ ]:
def make_continuous_data(n_periods=180, seed=7):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2024-01-01", periods=n_periods, freq="D")
    frames = []
    for offset, unique_id in enumerate(["region_A", "region_B"]):
        mean = 8 + 2 * offset + np.linspace(0, 2, n_periods) + 0.8 * np.sin(2 * np.pi * np.arange(n_periods) / 7)
        shape = 12.0
        y = rng.gamma(shape=shape, scale=mean / shape)
        frames.append(pd.DataFrame({"unique_id": unique_id, "ds": dates, "y": y}))
    return pd.concat(frames, ignore_index=True)

continuous_data = make_continuous_data()
continuous_data.head()

In [ ]:
continuous_fcst = MLForecast(
    models=[LinearRegression()], freq="D", lags=[1, 7, 14], date_features=["dayofweek"]
)
continuous_model = TwoStageForecasterWrapper(
    continuous_fcst, distribution=GammaFamily()
).fit(continuous_data, h=7, n_windows=4)

### Continuous PPF and CDF

Gamma quantiles are real-valued. For interval probabilities, subtract two CDF evaluations—for example, `CDF(12) - CDF(8)` gives the probability that demand is between 8 and 12.

In [ ]:
continuous_frame, continuous_distribution = continuous_model.predict_distribution(h=7)
continuous_ppf = continuous_distribution.ppf([0.05, 0.50, 0.95])

continuous_frame.assign(
    cdf_at_mean=continuous_distribution.cdf(continuous_frame["lambda_t"].to_numpy()),
    probability_between_8_and_12=continuous_distribution.cdf(12) - continuous_distribution.cdf(8),
    q05=continuous_ppf[:, 0],
    q50=continuous_ppf[:, 1],
    q95=continuous_ppf[:, 2],
).head()

### Optimize continuous inventory

The workflow is the same, but the Gamma PPF returns a continuous optimal quantity. Row-level decision costs are passed separately through `cost_df`; they are not forecast features.

In [ ]:
future_costs = continuous_fcst.make_future_dataframe(h=7)
future_costs["shortage_cost"] = np.where(
    future_costs["unique_id"] == "region_A", 6.0, 9.0
)
future_costs["holding_cost"] = 2.0

continuous_plan = NewsvendorOptimizer.optimize(
    continuous_frame,
    continuous_distribution,
    underage_cost="shortage_cost",
    overage_cost="holding_cost",
    cost_df=future_costs,
)
continuous_plan.head()

### Marginal benefit by inventory unit

For discrete distributions, `NewsvendorOptimizer.marginal_benefit` evaluates the expected net benefit of adding each inventory unit. Positive values favor stocking the unit; negative values indicate that its expected overage cost is larger than its shortage benefit. Use `max_k` to evaluate every unit from zero through an upper bound.

In [ ]:
marginal_benefits = NewsvendorOptimizer.marginal_benefit(
    count_frame,
    count_distribution,
    underage_cost=10.0,
    overage_cost=2.0,
    max_k=8,
)
marginal_benefits.head()

Use `units` instead when only specific inventory levels are relevant. The supplied order is preserved, and `max_k` and `units` are mutually exclusive. A stepped `range` provides a middle ground between a dense interval and a manually selected sparse grid.

In [ ]:
sparse_marginal_benefits = NewsvendorOptimizer.marginal_benefit(
    count_frame,
    count_distribution,
    underage_cost=10.0,
    overage_cost=2.0,
    units=[2, 5, 8],
)
sparse_marginal_benefits.head()

In [ ]:
stepped_marginal_benefits = NewsvendorOptimizer.marginal_benefit(
    count_frame,
    count_distribution,
    underage_cost=10.0,
    overage_cost=2.0,
    units=range(0, 21, 5),
)
stepped_marginal_benefits.head()

## Out-of-sample evaluation and calibration

Forecast diagnostics should use observations that were not used for fitting. The following example reserves the final 14 days, fits a separate model on the earlier history, and joins predictions to the held-out targets.

In [ ]:
evaluation_horizon = 14
cutoff = count_data["ds"].max() - pd.Timedelta(days=evaluation_horizon)
evaluation_train = count_data[count_data["ds"] <= cutoff].copy()
evaluation_test = count_data[count_data["ds"] > cutoff].copy()

evaluation_fcst = MLForecast(
    models=[LinearRegression()], freq="D", lags=[1, 7, 14], date_features=["dayofweek"]
)
evaluation_model = TwoStageForecasterWrapper(evaluation_fcst).fit(
    evaluation_train, h=7, n_windows=4
)
evaluation_predictions, evaluation_distribution = evaluation_model.predict_distribution(
    h=evaluation_horizon
)
evaluation_quantiles = evaluation_distribution.ppf([0.05, 0.50, 0.95])
for index, quantile in enumerate((0.05, 0.50, 0.95)):
    evaluation_predictions[f"q_{int(quantile * 100)}"] = evaluation_quantiles[:, index]

evaluation_results = evaluation_predictions.merge(
    evaluation_test[["unique_id", "ds", "y"]],
    on=["unique_id", "ds"],
    how="inner",
)
evaluation_results.head()

### Conditional-mean evaluation and calibration table

`FirstStageForecasterEvaluator.evaluate` summarizes point-forecast behavior. The calibration table groups similar predicted means and compares their average prediction with the average observed outcome. Well-calibrated bins have a `Mean_Residual` close to zero.

In [ ]:
mean_metrics = FirstStageForecasterEvaluator.evaluate(evaluation_results)
mean_calibration = FirstStageForecasterEvaluator.calibration_table(
    evaluation_results, n_bins=5
)

display(mean_metrics)
display(mean_calibration)

### Probabilistic evaluation

`TwoStageForecasterEvaluator.evaluate` reports pinball loss and empirical coverage for every requested quantile available in the forecast frame. A coverage gap close to zero indicates calibrated quantiles.

In [ ]:
probabilistic_metrics = TwoStageForecasterEvaluator.evaluate(
    evaluation_results, quantiles=(0.05, 0.50, 0.95)
)
probabilistic_metrics

## Save and restore with joblib

Persist the fitted wrapper—not only the underlying regressor—so the calibrated family and per-series dispersion parameters are restored together. Only load files from trusted sources because `joblib.load` can execute arbitrary code during deserialization.

In [ ]:
import joblib

model_path = "two_stage_forecaster.joblib"
joblib.dump(count_model, model_path)
restored_model = joblib.load(model_path)
restored_predictions, restored_distribution = restored_model.predict_distribution(h=7)
restored_ppf = restored_distribution.ppf([0.50, 0.95])
restored_predictions["q_50"] = restored_ppf[:, 0]
restored_predictions["q_95"] = restored_ppf[:, 1]
restored_predictions.head()

## Family comparison

| Target | Family | CDF | PPF | PMF | `NewsvendorOptimizer.optimize` output |
|---|---|---:|---:|---:|---|
| Non-negative integer counts | Negative Binomial (default) | Yes | Yes | Yes | Integer |
| Strictly positive continuous values | `GammaFamily()` | Yes | Yes | No | Continuous |

Always match the family to the target support: Negative Binomial rejects negative or non-integer targets, while Gamma rejects zero and negative targets.